# 1 — Extraction and Ingestion

Turns one PDF into indexed vectors, one stage at a time, with an inspection point
after each.

```
PDF ─▶ parse ─▶ inspect ─▶ figures ─▶ chunk ─▶ index
```

This notebook imports from `ingest.py` rather than restating it. `ingest.py` is the
same file that runs inside a Fargate task, so what you verify here is exactly what
runs in production — there is no second copy to drift out of sync.

| § | Stage | What to check |
|---|---|---|
| 0 | Setup | derived configuration |
| 1 | Parse | which enrichments applied |
| 2 | Inspect | every figure described, no undecoded formulas, no suspect tables |
| 3 | Read the extraction | the report, against the PDF |
| 4 | Figures | images stored (optional) |
| 5 | Chunk | table summaries produced, nothing truncated |
| 6 | Read the chunks | what will actually be embedded |
| 7 | Index | added / unchanged / removed |
| 8 | Verify | manifest written |

---
## 0. Setup

In [ ]:
%pip install -q -r requirements.txt

In [ ]:
import os

os.environ.setdefault("OPENAI_API_KEY", "")
os.environ.setdefault("PINECONE_API_KEY", "")

# Read once at import time, so set these before importing rag_common.
os.environ.setdefault("CHUNK_TOKEN_TARGET", "1024")     # chunk size, in tokens
os.environ.setdefault("FIGURE_AREA_THRESHOLD", "0.01")  # min page fraction for a figure
os.environ.setdefault("REPORT_DIR", "reports")

In [ ]:
import json
from pathlib import Path

import pandas as pd

import rag_common as rc
import ingest

SOURCE_PDF = Path("pdfs/AI-Enablers-Adopters-research-report.pdf")

# Set to your bucket to store figure images and share the parse cache.
# Leave as None to keep everything local.
BUCKET = None

DOC_ID = rc.slugify(SOURCE_PDF.stem)

print(f"document   : {DOC_ID}")
print(f"embedding  : {rc.EMBED_MODEL} ({rc.EMBED_DIMS}d)")
print(f"chunk size : {rc.CHUNK_TOKENS} tokens")
print(f"vision     : {rc.VISION_MODEL}")
print(f"reports    : {ingest.REPORT_DIR.resolve()}")

---
## 1. Parse

Docling produces a `DoclingDocument`: a typed object graph carrying tables, figures,
formulas, code blocks, reading order and page provenance. Chunking works on that
object, so structure never has to be recovered by pattern matching.

Almost every enrichment is off by default. Without `do_formula_enrichment` an
equation becomes a placeholder and its content is gone, with no error raised.

**This cell re-parses every time it runs**, which costs one vision call per figure —
seconds on this document, minutes on a long one. Run it once and keep `doc` in the
kernel; the cells below work on that object and are instant. Only come back here
when a parse setting actually changes.

There is no parse cache, deliberately. The output depends on the settings above as
much as on the file, so a cache keyed on the filename would return work done under
different settings — making a changed setting look like it did nothing.

In [ ]:
from docling.datamodel.pipeline_options import PdfPipelineOptions

# What this Docling build supports, and what it defaults to. Flag names move
# between releases, which is why ingest.py reads them rather than assuming.
for name, field in sorted(PdfPipelineOptions.model_fields.items()):
    if name.startswith(("do_", "generate_", "enable_")):
        print(f"  {name:36s} {field.default}")

In [ ]:
doc = ingest.parse_pdf(SOURCE_PDF)

In [ ]:
# Rendering the whole document to markdown is expensive on long PDFs, so do it once
# here and reuse it for the checks below and for the date heuristic.
markdown = doc.export_to_markdown()
doc_date = ingest.document_date(SOURCE_PDF, markdown[:4000])

print(f"pages {len(doc.pages)}   document date {doc_date}")

### What one element of each type looks like

Docling does not return a string. It returns a graph of typed pydantic objects, and
the type is what lets each be handled differently — a `TableItem` can be asked for
its cell grid, a `PictureItem` for its annotations and its rendered image.

The cell below reads the fields off the objects themselves rather than printing a
hand-picked set, so what you see is the real shape, including fields this pipeline
never touches. Three things worth looking for.

**`prov`** is a list, not a single value — one record per page the element appears
on, each with its own bounding box. That is why a chunk can carry `page` and
`page_end`.

**`data.table_cells`** on a `TableItem` is the grid before it is serialised: each
cell with its row and column spans and whether it is a header. `export_to_markdown()`
is a lossy rendering of this, which is what `table_looks_broken()` is checking.

**The Python class is not always what the label suggests.** `doc.iterate_items()`
yields real `TableItem` and `PictureItem` instances. But `chunk.meta.doc_items` —
which you will meet in the chunking section — holds lighter references that report
`label=TABLE` while failing an `isinstance` check. Testing the type there instead of
the label returns nothing, silently, for every chunk.

In [ ]:
# Every field on one sample of every element type the layout model found.
#
# Nothing here is hand-picked. The fields are read off the objects themselves, so
# what you see is the actual shape of the graph everything downstream works on —
# including fields this pipeline never touches, which is worth knowing when you go
# looking for something it does not currently use.

from docling_core.types.doc import PictureItem, TableItem


def preview(value, width: int = 88) -> str:
    """One line describing a field's value: its type, size, and a readable sample."""
    if value is None:
        return "None"
    if isinstance(value, (str, int, float, bool)):
        text = str(value).replace("\n", " ")
        return f"{text[:width]}{'…' if len(text) > width else ''}"
    if isinstance(value, (list, tuple)):
        if not value:
            return f"{type(value).__name__}[0] (empty)"
        inner = type(value[0]).__name__
        return f"{type(value).__name__}[{len(value)}] of {inner}"
    if isinstance(value, dict):
        return f"dict[{len(value)}] keys={list(value)[:6]}"
    return type(value).__name__


def dump_fields(obj, indent: str = "  ") -> None:
    """Print every declared field of a pydantic object, with its type and value.

    Docling's items are pydantic models, so `model_fields` is the authoritative
    list — better than dir(), which mixes in methods and validators. The fallback
    covers anything that is not a model.
    """
    fields = getattr(type(obj), "model_fields", None)
    if fields:
        for name in fields:
            value = getattr(obj, name, None)
            annotation = fields[name].annotation
            type_name = getattr(annotation, "__name__", str(annotation))
            print(f"{indent}{name:<18} {str(type_name)[:34]:<36} {preview(value)}")
    else:
        for name, value in vars(obj).items():
            if not name.startswith("_"):
                print(f"{indent}{name:<18} {type(value).__name__:<36} {preview(value)}")


seen = {}
for item, level in doc.iterate_items():
    label = str(getattr(item, "label", "?")).rsplit(".", 1)[-1]
    seen.setdefault(label, (item, level))

print(f"{len(seen)} element types found: {', '.join(sorted(seen))}\n")

for label in sorted(seen):
    item, level = seen[label]

    print("=" * 100)
    print(f"{label}   ({type(item).__name__}, tree level {level})")
    print("=" * 100)

    print("\nFIELDS")
    print(f"  {'name':<18} {'declared type':<36} value")
    print(f"  {'-'*18} {'-'*36} {'-'*40}")
    dump_fields(item)

    # prov is a list of provenance records, one per page the element appears on.
    # Its fields are where page numbers and bounding boxes actually live.
    prov = (getattr(item, "prov", None) or [None])[0]
    if prov is not None:
        print("\nprov[0]")
        dump_fields(prov, indent="  ")
        bbox = getattr(prov, "bbox", None)
        if bbox is not None:
            print("\n  prov[0].bbox")
            dump_fields(bbox, indent="    ")

    # Type-specific structure that the field list only hints at.
    if isinstance(item, TableItem):
        data = getattr(item, "data", None)
        if data is not None:
            print("\ndata  (the grid itself)")
            dump_fields(data, indent="  ")
            cells = getattr(data, "table_cells", None) or []
            if cells:
                print("\n  data.table_cells[0]")
                dump_fields(cells[0], indent="    ")
        markdown = item.export_to_markdown(doc)
        print(f"\nexport_to_markdown()  —  {len(ingest.table_cells(markdown))} cells, "
              f"structure {ingest.table_looks_broken(markdown) or 'looks sound'}")
        for line in markdown.splitlines()[:4]:
            print(f"  {line[:96]}")

    elif isinstance(item, PictureItem):
        annotations = ingest.picture_annotations(item)
        print(f"\nannotations  —  {len(annotations)}")
        for n, annotation in enumerate(annotations):
            print(f"\n  annotations[{n}]  ({type(annotation).__name__})")
            dump_fields(annotation, indent="    ")
        print(f"\nget_image(doc) : "
              f"{'a PIL image' if item.get_image(doc) is not None else 'None — not rendered'}")

    print()

---
## 2. Inspect the parse

Extraction failures do not raise. A disabled enrichment yields an empty annotation
or a placeholder, and every later stage runs happily on top of it — producing an
index that looks complete and is missing content. This is the checkpoint.

In [ ]:
report = ingest.inspect(doc, markdown)

**`pictures_described` should equal `pictures`.** If it is 0, figure descriptions
silently did nothing and every chart is invisible to search.

**`formulas_undecoded` should be 0** for any document containing equations.

**`tables_suspect` should be 0.** A non-zero value means TableFormer produced a grid
whose structure is wrong — row labels merged into value columns, or a header row
duplicated. That is worse than a missing table, because the summary describes a grid
that does not exist. Those tables are routed through the rendered image instead.

**`charts_with_data`** is how many figures yielded a numeric series rather than only
prose. Zero is common on vector-drawn charts and tells you `do_chart_extraction` is
costing a model pass for nothing on this corpus.

---
## 3. Read the extraction

`pages 7` tells you the file opened. It says nothing about whether the content is
good, and the parse cache is a JSON blob full of base64 images.

The report below is written **before any chunking or embedding**. Every element in
reading order, with page numbers, table markdown and figure descriptions inline, and
every failure marked with a banner rather than left as an absence — because an
absence is exactly what you fail to notice.

Open `reports/<doc>.extract.md` next to the PDF and read them side by side.

In [ ]:
report_path = ingest.write_extraction_report(doc, DOC_ID, SOURCE_PDF)

# The problems block sits at the top of the file, so this is the whole triage.
print(report_path.read_text()[:2000])

In [ ]:
# The same inventory as data, for checking a whole corpus at once rather than
# reading twenty reports.
inventory = json.loads(report_path.with_suffix(".json").read_text())

figures = [e for e in inventory["elements"] if e["kind"] == "FIGURE"]
tables = [e for e in inventory["elements"] if e["kind"] == "TABLE"]

print(f"{inventory['doc_id']}: {inventory['pages']} pages")
print(f"elements: {inventory['counts']}\n")
print(f"figures described : {sum(1 for f in figures if f.get('description'))}/{len(figures)}")
print(f"charts with data  : {sum(1 for f in figures if f.get('has_chart_data'))}/{len(figures)}")
print(f"tables serialised : {sum(1 for t in tables if t.get('cells', 0) > 0)}/{len(tables)}")
print(f"tables suspect    : {sum(1 for t in tables if t.get('structure_problems'))}/{len(tables)}")

if inventory["problems"]:
    print("\nproblems:")
    for problem in inventory["problems"]:
        print(f"  {problem}")
else:
    print("\nno extraction problems detected")

In [ ]:
# Read one figure description against the actual chart. This is the only check on
# whether the vision model read the numbers or wrote plausible prose about them.
for figure in figures:
    if figure.get("description"):
        print(f"FIGURE p{figure['page']}\n{figure['description']}\n")
        break

# And one table, with any structure warning attached.
for table in tables:
    if table.get("markdown"):
        print(f"TABLE p{table['page']}  {table['cells']} cells")
        if table.get("structure_problems"):
            print(f"  STRUCTURE SUSPECT: {'; '.join(table['structure_problems'])}")
        print(table["markdown"][:900])
        break

---
## 4. Figure images

The pixels are already rendered for the vision model. Storing them means an answer
can show the chart rather than only quote a description of it. Skipped when `BUCKET`
is `None`.

In [ ]:
figure_uris = ingest.save_figures(doc, DOC_ID, BUCKET)
print(f"{len(figure_uris)} figure images stored")

---
## 5. Chunk

`HybridChunker` splits on the document's own hierarchy, then refines against a token
budget. `contextualize()` prepends the heading path, and that string is what gets
embedded, so section context participates in retrieval.

### Tables get an extra chunk

A grid of numbers shares almost no vocabulary with a question like "how did
enrolment change" — those words appear in no cell. So every table produces one
additional chunk: a summary naming what it reports, its units, and the patterns that
hold across rows but appear nowhere in a single cell, like "every 4 weeks".

Retrieval matches the summary; the raw rows stay as separate chunks carrying the
exact values, linked by `table_id`.

When a table's parsed structure is unsound, the summary is generated from the
rendered image instead — describing a broken grid confidently is worse than not
describing it at all.

In [ ]:
records = ingest.build_records(doc, SOURCE_PDF, DOC_ID, doc_date, figure_uris)

In [ ]:
frame = pd.DataFrame([{
    "pos":      r["meta"]["position"],
    "type":     r["meta"]["content_type"],
    "tokens":   r["meta"]["n_tokens"],
    "page":     r["meta"]["page"],
    "table_id": r["meta"]["table_id"] or "",
    "source":   r["meta"].get("summary_source", ""),
} for r in records])

print(frame.groupby("type")["tokens"].agg(["count", "mean", "max"]).round(0).to_string())

n_summaries = int((frame["type"] == "table_summary").sum())
n_tables = frame.loc[frame.type == "table", "table_id"].nunique()
print(f"\ntable summaries : {n_summaries} for {n_tables} tables")
print(f"from image      : {int((frame['source'] == 'image').sum())}")
print(f"truncated       : {sum(1 for r in records if r['meta'].get('truncated'))}")
print(f"over budget     : {int((frame.tokens > rc.CHUNK_TOKENS).sum())}")

In [ ]:
# A summary and its fragments, in reading order. The summary should sit immediately
# before the rows it describes, and share their table_id.
summaries = [r for r in records if r["meta"]["content_type"] == "table_summary"]

if summaries:
    table_id = summaries[0]["meta"]["table_id"]
    for record in records:
        if record["meta"]["table_id"] == table_id:
            meta = record["meta"]
            source = f"  [{meta['summary_source']}]" if meta.get("summary_source") else ""
            print(f"[{meta['position']:>3}] {meta['content_type']:<14} p{meta['page']}{source}")
            print(f"      {record['text'][:200].replace(chr(10), ' ')}\n")
else:
    print("no table summaries in this document")

---
## 6. Read the chunks

The extraction report answers *did the parse work*. This one answers *is the string
being embedded the right string* — the exact text, its token count, and each table
summary beside the fragments it describes.

In [ ]:
chunk_report = ingest.write_chunk_report(records, DOC_ID)
print(chunk_report.read_text()[:1800])

In [ ]:
# One chunk of each type, read end to end.
for wanted in ("text", "table_summary", "table", "figure", "formula", "code"):
    record = next((r for r in records if r["meta"]["content_type"] == wanted), None)
    if record is None:
        continue
    meta = record["meta"]
    print("=" * 78)
    print(f"{meta['chunk_id']}  {wanted}  p{meta['page']}-{meta['page_end']}  "
          f"{meta['n_tokens']} tok")
    print(f"headings: {meta['headings']}")
    if meta.get("image_uri"):
        print(f"image: {meta['image_uri']}")
    print("-" * 78)
    print(record["text"][:600], "\n")

---
## 7. Index

Chunk identity is content-addressed: `{doc_id}:{sha256(text)[:16]}:{occurrence}`.

A positional key would change for every chunk after an edit, forcing a full
re-embed. A content hash changes only where the text changed, which turns
re-ingestion into a set difference. The `doc_id` prefix scopes the hash so identical
boilerplate in two documents cannot collide, and `occurrence` distinguishes text
that legitimately repeats within one document.

Because of that, re-running is idempotent: a duplicate run upserts identical vectors
and deletes nothing.

In [ ]:
index = rc.open_index(create=True)
plan = ingest.sync(index, DOC_ID, records)

In [ ]:
# Run this cell again: everything should be unchanged and nothing re-embedded.
ingest.sync(index, DOC_ID, records)

---
## 8. Verify

The manifest records how the index was built. The retrieval notebook reads it and
refuses to run if its own configuration disagrees — a mismatched embedding model
returns plausible rankings with no error, which is the failure hardest to notice.

In [ ]:
manifest = rc.write_manifest(DOC_ID, len(records),
                             extra={"source": SOURCE_PDF.name, "doc_date": doc_date})
print(json.dumps(manifest, indent=2))

Ingestion complete. Continue in the retrieval notebook.

To add another document, change `SOURCE_PDF` and re-run from §1. Each document
occupies its own key prefix and the manifest gains an entry.

The same code runs headless, and on AWS unchanged:

```bash
python ingest.py --pdf pdfs/report.pdf
python ingest.py --bucket B --key K
```